In [3]:
!pwd

/mnt/d/dev/git/learn/GAN


生成对抗网络（GANs）是当今计算机科学领域最有趣的想法之一。两个模型通过对抗过程同时训练。一个生成器（“艺术家”）学习创造看起来真实的图像，而判别器（“艺术评论家”）学习区分真假图像。

训练过程中，生成器在生成逼真图像方面逐渐变强，而判别器在辨别这些图像的能力上逐渐变强。当判别器不再能够区分真实图片和伪造图片时，训练过程达到平衡。

# 生成式模型与判别式模型
## 1. 判别式模型
判别式模型，即 Discriminative Model，又被称为条件概率模型，它估计的是条件概率分布(conditional distribution)， p(class|context) 。

举个例子，我们给定(x,y)对，4个样本。(1,0), (1,0), (2,0), (2, 1)，p(y|x)是事件x发生时y的条件概率，它的计算如下：  
![判别式模型](./imgs/discriminativemodel.jpeg)   

## 2. 生成式模型
即 Generative Model ，生成式模型 ，它估计的是联合概率分布（joint probability distribution），p(class,context)=p(class|context)*p(context) 。p(x,y)，即事件x与事件y同时发生的概率。同样以上面的样本为例，它的计算如下：   
![判别式模型](./imgs/generativemodel.jpeg)  

## 3. 常见模型
常见的判别式模型有 Logistic Regression，Linear Regression，SVM，Traditional Neural Networks Nearest Neighbor，CRF 等。

常见的生成式模型有 Naive Bayes，Mixtures of Gaussians， HMMs，Markov Random Fields 等。

## 4. 判别式模型与生成式模型比较
判别式模型 ，优点是分类边界灵活 ，学习简单，性能较好 ；缺点是不能得到概率分布 。

生成式模型 ，优点是收敛速度快，可学习分布，可应对隐变量 ；缺点是学习复杂 ，分类性能较差。   
![判别式模型和生成式模型比较](./imgs/discriminativemodel_diag.jpeg)  
上面是一个分类例子，可知判别式模型，有清晰的分界面，而生成式模型，有清晰的概率密度分布。生成式模型，可以转换为判别式模型，反之则不能。

# GAN的基本原理
GAN，即Generative adversarial net，它同时包含判别式模型和生成式模型，一个经典的网络结构如下：   
![GAN定义](./imgs/gan_diag.jpeg)  

## 1. 基本原理   
![GAN原理](./imgs/gan_diag1.jpeg)   

GAN的原理很简单，它包括两个网络，一个生成网络，不断生成数据分布。一个判别网络，判断生成的数据是否为真实数据。上图是原理展示，黑色虚线是真实分布，绿色实线是生成模型的学习过程，蓝色虚线是判别模型的学习过程，两者相互对抗，共同学习到最优状态。

## 2. 优化目标与求解
下面是它的优化目标。
![优化目标与求解-1](./imgs/formula1.jpeg)   


D是判别器，它的学习目标，是最大化上面的式子，而G是生成器，它的学习目标，是最小化上面的式子。上面问题的求解，通过迭代求解D和G来完成。

要求解上面的式子，等价于求解下面的式子。

![优化目标与求解-2](./imgs/formula2.jpeg)  

其中D(x)属于(0,1)，上式是alog(y) + blog(1−y)的形式，取得最大值的条件是D(x)=a/(a+b)，此时等价于下面式子。

![优化目标与求解-3](./imgs/formula3.jpeg)  

如果用KL散度来描述，上面的式子等于下面的式子。    
![优化目标与求解-4](./imgs/formula4.jpeg)  

当且仅当pdata(x)=pg(x)时，取得极小值-log4，此时d=0.5，无法分辨真实样本和假样本。

GAN从理论上，被证实存在全局最优解。


## 3. 如何训练
![如何训练](./imgs/howtotrain.jpeg)    
就是采用判别式模型和生成式模型分别循环依次迭代的方法，与CNN一样，使用梯度下降来优化。

## 4. GAN的主要问题
GAN从本质上来说，有与CNN不同的特点，因为GAN的训练是依次迭代D和G，如果判别器D学的不好，生成器G得不到正确反馈，就无法稳定学习。如果判别器D学的太好，整个loss迅速下降，G就无法继续学习。

GAN的优化需要生成器和判别器达到纳什均衡，但是因为判别器D和生成器G是分别训练的，纳什平衡并不一定能达到，这是早期GAN难以训练的主要原因。另外，最初的损失函数也不是最优的

# GAN的应用
## 1. 数据生成
* 从GAN到Conditional GAN
GAN的生成式模型可以拟合真实分布，所以它可以用于伪造数据。DCGAN【3】是第一个用全卷积网络做数据生成的，下面是它的基本结构和生成的数据。   
![conditional GAN](./imgs/cgan.jpeg)
![DC GAN](./imgs/dcgan.jpeg)

输入100维的噪声，输出64×64的图像，从mnist的训练结果来看，还不错。

但是它的问题是不能控制生成的数字是1还是9，所以后来有了CGAN【4】,即条件GAN，网络结构如下。   
![条件GAN结构](./imgs/cgan_structure.jpeg)

它将标签信息encode为一个向量，串接到了D和G的输入进行训练，优化目标发生了改变。

![从GAN到Conditional_GAN-4](./imgs/gan4.jpeg)

与cgan类似，infogan【5】将噪声z进行了拆解，一是不可压缩的噪声z，二是可解释的隐变量c，可以认为infogan就是无监督的cgan，这样能够约束c与生成数据之间的关系，控制一些属性，比如旋转等。   

![infogan](./imgs/cgan2.jpeg)

条件GAN的出现，使得控制GAN的输出有了可能，出现了例如文本生成图像【6】的应用。   
![text to image](./imgs/text2img.jpeg)

* 金字塔GAN   
  原始的GAN生成图的分辨率太小，无法实用，借鉴经典图像中的金字塔算法，LAPGAN【7】/StackedGAN8【8】各自提出类似的想法，下面是LAPGAN的结构。   
  ![金字塔GAN](./imgs/egypt_gan.jpeg)   
  ![金字塔GAN2](./imgs/egypt_gan2.jpeg)

  它有以下特点。

(1)使用残差逼近，学习相对容易。

(2)逐级独立训练提高了网络简单记忆输入样本的难度，减少了每一次 GAN 需要学习的内容，也就从而增大了 GAN 的学习能力和泛化能力。

在这个基础上，nvidia-gan【9】生成了1024分辨率的图片，它的网络结构和生成结果如下。

 ![金字塔GAN2](./imgs/egypt_gan3.jpeg)    
  ![金字塔GAN2](./imgs/egypt_gan4.jpeg)

  * cross domain学习

    cross domain的学习，提供了更丰富的数据生成应用。

在传统的domain adaption中，我们需要学习或者训练一个domain adaptor，而这个domain adaptor需要用source domain和对应的target domain的训练图片来训练。coGAN【10】/align gan【11】可以在两个domain不存在对应样本的情况下学出一个联合分布，方法是每一个domain使用一个GAN，并且将高层的语义信息进行强制权值共享。   

  ![cross domain1](./imgs/crossdomain1.jpeg)
  
  ![cross domain2](./imgs/crossdomain2.jpeg)   
  在这样的基础上，有一些很有意义的应用。比如苹果simGAN【12】用于优化仿真数据的方案，此时生成器G的输入是合成图像，而不是随机向量，它完美学习到了人工合成图片（synthetic images）数据分布到真实图片（real images）数据分布的映射。


  ![cross domain3](./imgs/crossdomain3.jpeg)   
  下面是生成的结果，很有工程意义。   
  ![cross domain4](./imgs/crossdomain4.jpeg)

* 一些很酷的应用    
  reative-gan【13】，用于生成艺术风格的图片。    
![sample](./imgs/sample1.jpeg)      



![sample](./imgs/sample2.jpeg)   


DesignGan【14】，用于设计T恤。    

![sample](./imgs/sample3.jpeg)

![sample](./imgs/sample4.jpeg)    

P-GAN【15】，用于人脸正脸化。

![sample](./imgs/sample5.jpeg)

![sample](./imgs/sample6.jpeg)

## 2. 风格迁移
文【20】实现了像素级别的风格转换，它的关键是提供了两个域中有相同数据的成对训练样本，本质上，是一个CGAN。   


![sample](./imgs/sample7.jpeg)   

cycle-gan【21】/dual-gan【22】则更胜一筹，不需要配对的数据集，可以实现源域和目标域的相互转换。


![sample](./imgs/sample8.jpeg)    

pairedcycle【23】，将源域和目标域的相互转换用到化妆和去妆，很有趣的应用。    
![sample](./imgs/sample9.jpeg)    

文【24】学习了一个数据集到另一个数据集的迁移，可以用于迁移学习，如实现漫画风格。    

![sample](./imgs/sample10.jpeg)

![sample](./imgs/sample11.jpeg)    

文【25】实现了动作的迁移。

![sample](./imgs/sample12.jpeg)   

文【26】实现了年龄的仿真。

![sample](./imgs/sample13.jpeg)    

文【27】提出了一种去雨的算法，很有实际意义。
![sample](./imgs/sample14.jpeg)    

文【28】实现了卡通风格的转换。   

![sample](./imgs/sample15.jpeg)    

文【29】实现了字体风格的迁移。

![sample](./imgs/sample16.jpeg)     

文【30】实现了去模糊。

![sample](./imgs/sample17.jpeg)




## 3. 超分辨重建
srgan【31】是最早使用GAN做超分辨重建的应用，它将输入从随机噪声改为低分辨率的图片，使用了残差结构和perception loss，有很大的应用价值。    

![sample](./imgs/sample18.jpeg)    

![sample](./imgs/sample19.jpeg)    

超分辨重建可用于小脸的检测【32】，是个值得关注的方向。

![sample](./imgs/sample20.jpeg)

